In [ ]:
import os, random, warnings, gc, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 16
IMG_SIZE = 384
np.random.seed(42); torch.manual_seed(42); random.seed(42)
print(f"Device: {DEVICE}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ======================================================================
# ANTI-SHORTCUT DATASET (NO U-Net, NO segmented data)
# ======================================================================
# Training on RAW images only.
# Standard augmentation: Resize, Crop, Flip, Rotate, ColorJitter, Affine
# Mixup for regularization
# ======================================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
MIXUP_ALPHA = 0.4

train_transform = T.Compose([
    T.Resize((416, 416)),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.RandomAdjustSharpness(sharpness_factor=1.5, p=0.3),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class AntiShortcutDataset(Dataset):
    def __init__(self, paths, labels, class_names, augment=False, use_mixup=0.0):
        self.paths = paths
        self.labels = [class_names.index(l) for l in labels]
        self.class_names = class_names
        self.augment = augment
        self.use_mixup = use_mixup

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
        except:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        label = self.labels[idx]

        if self.augment:
            img_tensor = train_transform(img)

            if random.random() < self.use_mixup:
                j = random.randint(0, len(self.paths) - 1)
                if j != idx:
                    try:
                        img2 = Image.open(self.paths[j]).convert("RGB")
                    except:
                        img2 = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
                    img2_tensor = train_transform(img2)
                    lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                    mixed = lam * img_tensor + (1 - lam) * img2_tensor
                    label2 = self.labels[j]
                    return mixed, label, label2, lam
        else:
            img_tensor = val_transform(img)

        return img_tensor, label, label, 1.0


def collate_fn(batch):
    imgs = torch.stack([b[0] for b in batch])
    y1 = torch.tensor([b[1] for b in batch])
    y2 = torch.tensor([b[2] for b in batch])
    lam = torch.tensor([b[3] for b in batch])
    return imgs, y1, y2, lam

print("Dataset modules loaded (NO U-Net, NO segmented data).")

In [ ]:
# ======================================================================
# DATA LOADING - chest-xray-pneumonia + kostas (RAW images)
# ======================================================================
chest_xray_paths = {
    "cx": ["/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray",
           "/kaggle/input/chest-xray-pneumonia/chest_xray",
           "/kaggle/input/paultimothymooney-chest-xray-pneumonia/chest_xray"],
    "kostas": ["/kaggle/input/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal",
               "/kaggle/input/chest-xrays-bacterial-viral-pneumonia-normal",
               "/kaggle/input/kostasdiamantaras-chest-xrays-bacterial-viral-pneumonia-normal"]
}

cx_normal, cx_bact, cx_viral = [], [], []
CHEST_XRAY = None
for p in chest_xray_paths["cx"]:
    if os.path.isdir(p):
        CHEST_XRAY = p
        break
if not CHEST_XRAY:
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "chest_xray" and os.path.isdir(os.path.join(root, "train", "NORMAL")):
            CHEST_XRAY = root
            break

if CHEST_XRAY:
    print(f"chest-xray-pneumonia at: {CHEST_XRAY}")
    for split in ["train", "val"]:
        norm_dir = os.path.join(CHEST_XRAY, split, "NORMAL")
        pneu_dir = os.path.join(CHEST_XRAY, split, "PNEUMONIA")
        if os.path.isdir(norm_dir):
            for f in os.listdir(norm_dir):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    cx_normal.append(os.path.join(norm_dir, f))
        if os.path.isdir(pneu_dir):
            for f in os.listdir(pneu_dir):
                if f.lower().endswith((".jpeg", ".jpg", ".png")):
                    fp = os.path.join(pneu_dir, f)
                    if "bacteria" in f.lower(): cx_bact.append(fp)
                    elif "virus" in f.lower(): cx_viral.append(fp)
    print(f"CX: NORMAL={len(cx_normal)}, BACT={len(cx_bact)}, VIRAL={len(cx_viral)}")
else:
    print("WARNING: chest-xray-pneumonia NOT FOUND")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "NORMAL" in dirs and "PNEUMONIA" in dirs:
            print(f"  Found candidate: {root}")

KOSTAS = None
for p in chest_xray_paths["kostas"]:
    if os.path.isdir(p):
        KOSTAS = p
        break
if not KOSTAS:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "labels_train.csv" in files:
            KOSTAS = root
            break

k_normal, k_bact, k_viral = [], [], []
if KOSTAS:
    print(f"\nkostas at: {KOSTAS}")
    labels_csv = os.path.join(KOSTAS, "labels_train.csv")
    if os.path.exists(labels_csv):
        kostas_df = pd.read_csv(labels_csv)
        kostas_df.columns = [c.strip() for c in kostas_df.columns]
        for root, dirs, files in os.walk(KOSTAS):
            for f in files:
                if f.lower().endswith((".png", ".jpg", ".jpeg")):
                    fp = os.path.join(root, f)
                    match = kostas_df.loc[kostas_df["file_name"] == f, "class_id"]
                    if len(match) > 0:
                        cls = int(match.iloc[0])
                        if cls == 0: k_normal.append(fp)
                        elif cls == 1: k_bact.append(fp)
                        elif cls == 2: k_viral.append(fp)
    print(f"Kostas: NORMAL={len(k_normal)}, BACT={len(k_bact)}, VIRAL={len(k_viral)}")
else:
    print("WARNING: kostas NOT FOUND")

normal_pool = cx_normal + k_normal
bact_pool = cx_bact + k_bact
viral_pool = cx_viral + k_viral
random.shuffle(normal_pool); random.shuffle(bact_pool); random.shuffle(viral_pool)

CLS3 = ["NORMAL", "BACTERIAL", "VIRAL"]
all_paths = list(normal_pool) + list(bact_pool) + list(viral_pool)
all_labels = ["NORMAL"]*len(normal_pool) + ["BACTERIAL"]*len(bact_pool) + ["VIRAL"]*len(viral_pool)
print(f"\nCOMBINED: NORMAL={len(normal_pool)}, BACTERIAL={len(bact_pool)}, VIRAL={len(viral_pool)}")
print(f"Total: {len(all_paths)} | {dict(Counter(all_labels))}")

In [ ]:
# ======================================================================
# 3-CLASS TRAINING: Normal vs Bacterial vs Viral (anti-shortcut, NO U-Net)
# ======================================================================
# Single EfficientNet-B2@384 classifier: NORMAL / BACTERIAL / VIRAL
# ======================================================================

tr_paths, va_paths, tr_labels, va_labels = train_test_split(
    all_paths, all_labels, test_size=0.12, stratify=all_labels, random_state=42
)
print(f"Train: {Counter(tr_labels)}  Val: {Counter(va_labels)}")

tr_ds = AntiShortcutDataset(tr_paths, tr_labels, CLS3, augment=True, use_mixup=0.4)
va_ds = AntiShortcutDataset(va_paths, va_labels, CLS3, augment=False)

cw = Counter(tr_labels)
w = [1.0 / cw[x] for x in tr_labels]
tr_loader = DataLoader(tr_ds, BATCH_SIZE, sampler=WeightedRandomSampler(w, len(w), True),
                       num_workers=2, pin_memory=True, collate_fn=collate_fn)
va_loader = DataLoader(va_ds, BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
                       collate_fn=collate_fn)
print("Data loaders ready.")

def make_model(n):
    m = models.efficientnet_b2(weights="IMAGENET1K_V1")
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n))
    return m

model = make_model(len(CLS3)).to(DEVICE)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

head_params = list(model.classifier.parameters())
backbone_params = [p for p in model.parameters() if id(p) not in set(id(p) for p in head_params)]

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": 1e-4},
    {"params": head_params, "lr": 1e-3},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60)

best_acc, patience_cnt, PATIENCE = 0, 0, 15
WEIGHTS_PATH = "/kaggle/working/chest_xray_3class.pth"

print(f"\n{'='*60}")
print(f"3-CLASS TRAINING: Normal/Bacterial/Viral @{IMG_SIZE}px (RAW images)")
print(f"{'='*60}")

for ep in range(1, 61):
    model.train()
    tl = 0
    for imgs, y1_batch, y2_batch, lam_batch in tr_loader:
        imgs = imgs.to(DEVICE)
        y1_b = y1_batch.to(DEVICE)
        y2_b = y2_batch.to(DEVICE)
        lam_b = lam_batch.to(DEVICE).float()
        logits = model(imgs)
        loss = (lam_b * criterion(logits, y1_b) + (1 - lam_b) * criterion(logits, y2_b)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        tl += loss.item()

    model.eval()
    pr, tr_ = [], []
    with torch.no_grad():
        for imgs, y1_batch, _, _ in va_loader:
            imgs = imgs.to(DEVICE)
            pr.extend(model(imgs).argmax(1).cpu().numpy())
            tr_.extend(y1_batch.numpy())
    acc = accuracy_score(tr_, pr)
    print(f"E{ep:3d} TL:{tl/len(tr_loader):.4f} VA:{acc:.4f}")
    scheduler.step()
    if acc > best_acc:
        best_acc = acc
        patience_cnt = 0
        torch.save(model.state_dict(), WEIGHTS_PATH)
        print(f"  -> Saved (new best)")
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stop @ {ep}")
            break

print(f"\nBest val acc: {best_acc:.4f}")
model.load_state_dict(torch.load(WEIGHTS_PATH))
model.eval()
pr, tr_ = [], []
with torch.no_grad():
    for imgs, y1_batch, _, _ in va_loader:
        imgs = imgs.to(DEVICE)
        pr.extend(model(imgs).argmax(1).cpu().numpy())
        tr_.extend(y1_batch.numpy())
print("\nClassification Report (val):")
print(classification_report(tr_, pr, target_names=CLS3, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(tr_, pr))

In [ ]:
# ======================================================================
# EVAL: TTA (Test-Time Augmentation)
# ======================================================================
def predict_loader(model, loader, tta=True):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for imgs, y1_b, _, _ in loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)
            if tta:
                out = out + model(torch.flip(imgs, dims=[3]))
            p = torch.softmax(out, dim=1)
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(y1_b.numpy())
            probs.extend(p.cpu().numpy())
    return np.array(preds), np.array(trues), np.array(probs)

eval_model = make_model(len(CLS3)).to(DEVICE)
eval_model.load_state_dict(torch.load(WEIGHTS_PATH))
eval_model.eval()

print("="*60)
print("3-CLASS - VAL TTA")
print("="*60)
all_preds, all_true, all_probs = predict_loader(eval_model, va_loader, tta=True)
print(classification_report(all_true, all_preds, target_names=CLS3, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(all_true, all_preds))

try:
    auc = roc_auc_score(all_true, all_probs, multi_class="ovr", average="macro")
    print(f"Macro AUC: {auc:.4f}")
except: pass

for i, cls in enumerate(CLS3):
    mask = all_true == i
    print(f"{cls} recall: {(all_preds[mask] == i).mean():.4f}")

print(f"\n{'='*60}")
print("FLASK WEIGHTS SAVED")
print("="*60)
w_size = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f"3-class model: chest_xray_3class.pth ({w_size:.2f} MB)")